# REQUIREMENTS

CREATE A 🔑 SECRET NAMED NGROK_KEY

https://dashboard.ngrok.com/

#INSTRUCTIONS
Use T4 GPU on runtime type and latest runtime version

Run All and wait for NGROK LINK

In [ ]:
# @title Requirement Check
# @markdown Allow Access for gdrive and secrets
from google.colab import drive, userdata, auth
from googleapiclient.discovery import build
from IPython.display import clear_output
import os
import sys
#DRIVE SETUP
drive.mount('/content/drive')

SHORTCUT_NAME = "BA-CHATBOT"
SHORTCUT_TARGET_PATH = f"/content/drive/MyDrive/{SHORTCUT_NAME}"
PUBLIC_FOLDER_ID = "1lKNUGjIwZhKwu_htckR8hAsjh6S-SMUj"

if not os.path.exists(SHORTCUT_TARGET_PATH):
    print(f"'{SHORTCUT_NAME}' Shortcut not detected in My Drive. Initiating automatic link creation...")
    auth.authenticate_user()
    try:
        drive_service = build('drive', 'v3')
        shortcut_metadata = {
            'name': SHORTCUT_NAME,
            'mimeType': 'application/vnd.google-apps.shortcut',
            'shortcutDetails': {
                'targetId': PUBLIC_FOLDER_ID
            }
        }
        file = drive_service.files().create(body=shortcut_metadata, fields='id').execute()
        drive.mount('/content/drive', force_remount=True)
    except Exception as e:
        print(f"Failed to create shortcut via API: {e}")
else:
    print(f"Shortcut verified")

#NGROK CHECK
NGROK_KEY = userdata.get('NGROK_KEY')
if not NGROK_KEY or not str(NGROK_KEY).strip():
  print("NGROK_KEY NOT FOUND")
  sys.exit("Execution stopped: NGROK_KEY is required.")
else:
  print("NGROK_KEY is found")

In [ ]:
# @title INITIAL SETUP
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install flask flask-cors requests pillow pyngrok
!git clone https://github.com/modelscope/DiffSynth-Studio.git
%cd DiffSynth-Studio
!pip install -e .
!pip install modelscope torch torchvision transformers accelerate

clear_output(wait=True)
print("FINISHED INSTALLING")

In [ ]:
# @title  MODEL SETUP
import torch
import random
from diffsynth.pipelines.anima_image import AnimaImagePipeline, ModelConfig
from llama_cpp import Llama

filePaths = "/content/drive/MyDrive/BA-CHATBOT/Anima"
loraPaths = f"{filePaths}/Loras"
MODEL_PATH = f"{filePaths}/llama 8b.gguf"

vram_config = {
    "offload_dtype": torch.float16,
    "offload_device": "cuda",
    "onload_dtype": torch.float16,
    "onload_device": "cuda",
    "preparing_dtype": torch.float16,
    "preparing_device": "cuda",
    "computation_dtype": torch.float16,
    "computation_device": "cuda",
}

pipe = AnimaImagePipeline.from_pretrained(
    torch_dtype=torch.float16,
    device="cuda",
    model_configs=[
        ModelConfig(path=f"{filePaths}/anima-base-v1.0.safetensors", **vram_config),
        ModelConfig(path=f"{filePaths}/qwen_3_06b_base.safetensors", **vram_config),
        ModelConfig(path=f"{filePaths}/qwen_image_vae.safetensors", **vram_config),
    ],
    tokenizer_config=ModelConfig(path=f"{filePaths}/Qwen/"),
    tokenizer_t5xxl_config=ModelConfig(path=f"{filePaths}/StableDiffusion/"),
    vram_limit=torch.cuda.mem_get_info()[1] / (1024 ** 3) - 0.5,
)

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=8192,
    n_gpu_layers=-1,
    chat_format="llama-3"
)

clear_output(wait=True)
print("MODELS LOADED")

In [ ]:
#@title FUNCTIONS AND CONFIGURATIONS
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
import json
import time
import base64
from io import BytesIO
from pyngrok import ngrok

SYSTEM_PROMPT_FILE = f"{filePaths}/TaggerPrompt.txt"
with open(SYSTEM_PROMPT_FILE, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

NGROK_KEY = userdata.get('NGROK_KEY')
ngrok.set_auth_token(NGROK_KEY)

#CHARACTERS
character = ["Kasumi", "Natsu", "Kei"]
current_character = character[0]

character_tags = {
    "Kasumi" : {
        "tag" : "kinugawa_kasumi",
        "outfit" : ["white coat", "red shirt", "black shorts"]
    },
    "Natsu" : {
        "tag" : "Yutori_natsu",
        "outfit" : ["school uniform", "blue skirt"]
    },
    "Kei" : {
        "tag" : "tendou_kei",
        "outfit" : ["school uniform", "blue skirt"]
    }
}

#STYLES
style_path = f"{loraPaths}/style/"
style_config = {
    "DEFAULT" : {
        "safetensor" : "Default.safetensors",
        "trigger" : "@BlueArchStyle"
    },
    "CLEAR" : {
        "safetensor" : "Clear - Vibestyle.safetensors",
        "trigger" : ""
    },
    "PASTEL" : {
        "safetensor" : "Pastel - VibeStyle.safetensors",
        "trigger" : ""
    },
    "CHIBI" : {
        "safetensor" : "Chibi - VibeStyle.safetensors",
        "trigger" : ""
    },
    "BISHOUJO" : {
        "safetensor" : "Bishoujo - VibeStyle.safetensors",
        "trigger" : ""
    },
    "FLAT COLOR" : {
        "safetensor" : "flat_color.safetensors",
        "trigger" : "flat color, no lineart"
    },
    "RL" : {
        "safetensor" : "RL.safetensors",
        "trigger" : ""
    }
 }

current_style = style_config["DEFAULT"]

def change_lora(selected_char, selected_style):
  pipe.clear_lora()
  turbo_lora = ModelConfig(path=f"{loraPaths}/anima-turbo-lora-v0.1.safetensors")
  pipe.load_lora(pipe.dit, turbo_lora, alpha=1)
  character_lora = ModelConfig(path=f"{loraPaths}/{selected_char}V1FIXED.safetensors")
  pipe.load_lora(pipe.dit, character_lora, alpha=1.0)
  style_lora = ModelConfig(path=f"{style_path}{selected_style['safetensor']}")
  pipe.load_lora(pipe.dit, style_lora, alpha=1.0)

change_lora(current_character, current_style)

def generate_image(prompt):
    negative_prompt = "worst quality, low quality, monochrome, zombie, interlocked fingers, Aissist, cleavage, logo, text, speech bubble, chat"

    seed = random.randint(1, 10_000_000)

    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        cfg_scale=1.0,
        seed=seed,
        num_inference_steps=20,
        height=512,
        width=512
    )
    return image

def build_danbooru_prompt(chat, character_name, current_scene, player_name="Sensei"):
    lines = []

    for msg in chat[-7:-2]:
        role = msg.get("role")
        content = msg.get("content", "").strip()
        if not content:
            continue

        if role == "user":
            lines.append(f"{player_name}: {content}")
        else:
            lines.append(f"{character_name}: {content}")

    prompt = f"""
    [CONVERSATION HISTORY]
    {"\n".join(lines)}

    USE THIS AS HISTORY, JUST SAY 'SKIPPED' FOR NOW
    """
    recent_npc_msg = chat[-1].get("content", "").strip()
    recent_sensei_msg = ""
    if len(chat) >= 2:
      recent_sensei_msg = chat[-2].get("content", "").strip()


    recent_prompt = f"""
    [LAST SCENE]
    {str(current_scene)}
    [CURRENT CHAT]
    {player_name} : {recent_sensei_msg}
    {character_name} : {recent_npc_msg}
    """

    return prompt, recent_prompt

clear_output(wait=True)
print("CONFIGURATIONS READY")

In [ ]:
#@title SERVER SETUP
#@markdown #RUN AND WAIT FOR THE NGROK URL
last_scene = {
    "subject" : "1girl, pov, solo focus",
    "location": "",
    "action": "",
    "mood": "",
    "time": "",
    "outfit": character_tags[current_character]["outfit"],
    "objects": []
}
chat_history = [

]

request_active = False


app = Flask(__name__)
CORS(app)
@app.route("/character/select", methods=["POST"])
def select_character():
    global current_character
    global last_scene
    global chat_history
    global current_style
    global request_active

    if request_active:
      return jsonify({
        "status": "busy"
      }), 429
    request_active = True

    try:
      data = request.json

      selected_character = data.get("character")
      selected_style = data.get("style")

      chat_history = []
      current_history = data.get("chat", [])

      for i in current_history:
        i.pop("index", None)
        chat_history.append(i)

      if selected_character not in character_tags:
          return jsonify({
              "status": "error",
              "message": "Invalid character"
          }), 400

      current_character = selected_character

      # Reset scene memory
      last_scene = {
          "subject": "1girl, pov, solo focus",
          "location": "",
          "action": "",
          "mood": "",
          "time": "",
          "outfit": character_tags[current_character]["outfit"],
          "objects": []
      }
      print(f"CHARACTER CHANGED \nCurrent Character: {current_character}")
      current_style = style_config[selected_style]
      change_lora(current_character, current_style)


      return jsonify({
          "status": "ok",
          "character": current_character,
          "scene": last_scene
      })
    except Exception as e:
      print("ERROR:", e)
      return jsonify({
          "status": "error",
          "message": str(e)
      }), 500
    finally:
      request_active = False



@app.route("/generate", methods=["POST"])
def generate():
    global last_scene
    global chat_history
    global current_style
    global request_active


    if request_active:
      return jsonify({
        "status": "busy"
      }), 429
    request_active = True

    try:
      data = request.json
      incoming_scene = data.get("scene", {})
      incoming_history = data.get("chat", [])

      current_index = 0

      for i in incoming_history:
        i.pop("index", None)
        if i not in chat_history:
          if len(incoming_history) <= 2:
            chat_history.append(i)
          else:
            chat_history.insert(current_index,i)
            current_index += 1
        else:
          current_index = chat_history.index(i) + 1


      for key, value in incoming_scene.items():

        if key in ["outfit", "objects"]:

            if isinstance(value, str):

                value = [
                    item.strip()
                    for item in value.split(",")
                    if item.strip()
                ]

        if value is not None and value != "":
            last_scene[key] = value


      # ---------------- BUILD PROMPT ----------------
      prompt, recent_prompt = build_danbooru_prompt(chat_history, current_character, last_scene)
      print("Waiting for llama tags...")
      history_list = [
          {"role" : "system", "content" : SYSTEM_PROMPT},
          {"role" : "user", "content" : prompt},
          {"role" : "assistant", "content" : "SKIPPED"},
          {"role" : "user", "content" : recent_prompt}
      ]
      #LLAMA
      response = llm.create_chat_completion(
          messages=history_list,
          temperature=0.2,
          top_p=0.95,
          max_tokens=2000,
          repeat_penalty=1.05,
          response_format={
              "type": "json_object"
          }
      )

      tags = response["choices"][0]["message"]["content"]
      data_dict = json.loads(tags)
      current_scene = data_dict.get("scene", {})


      history_list.append({"role" : "assistant", "content" : tags})
      cleaner_prompt = f"""
      You are a natural prompt cleaner. Your job is to rewrite the given natural prompt to be extremely short and concise, following strict rules.

      RULES:
      1. The character is '{current_character}'. Always refer to her as 'she'.
      2. Refer to Sensei as 'the viewer'. Do not use names or first person.
      3. Output ONLY ONE sentence, maximum 15 words.
      4. Describe ONLY ONE visible action. No sequences (remove 'then', 'before', 'after', 'first', 'next').
      5. Remove all internal thoughts, feelings, and dialogue (e.g., 'she feels', 'she thinks').
      6. Remove any descriptions of aftermath or completed actions (e.g., 'has already', 'had placed').
      7. If the prompt contains multiple actions, pick ONLY the most memorable action that is directed toward the viewer (handing, looking, reaching, speaking – without quoting words, or the viewer touching her).
      8. Keep only what can be seen in a single image (pose, expression, hand motion, gaze).
      9. Do NOT add any new content not present in the original prompt.
      10. Output ONLY the cleaned natural prompt text. No extra words, no quotes, no labels.

      PREVIOUS NATURAL_PROMPT:
      {data_dict.get("natural_prompt")}
      """

      history_list.append({"role" : "user", "content": cleaner_prompt})

      cleaned_response = llm.create_chat_completion(
          messages=history_list,
          temperature=0.2,
          top_p=0.95,
          max_tokens=2000,
          repeat_penalty=1.05
      )

      clean_natural_prompt = cleaned_response["choices"][0]["message"]["content"]

      print(f"PREVIOUS NATURAL PROMPT: {data_dict.get("natural_prompt")}")
      print(f"CLEANED NATURAL PROMPT : {clean_natural_prompt}")

      for k, v in current_scene.items():
          if v is not None and v != "":
              last_scene[k] = v

      current_tags = data_dict.get("tags")
      meta_tags = ["1girl", "solo focus", "pov"]
      for tag in meta_tags:
        if not tag in current_tags:
          current_tags.append(tag)

      style_trigger = current_style.get("trigger", "")
      tag_prompt = f'{character_tags[current_character]["tag"]}, {style_trigger}, {", ".join(current_tags)} \n{clean_natural_prompt}'

      print("\n================ TAGS ================\n")
      print(tag_prompt)
      print("\n=============================================\n")

      # ---------------- IMAGE GENERATION ----------------
      image = generate_image(tag_prompt)
      buffer = BytesIO()
      image.save(buffer, format="PNG")
      buffer.seek(0)
      encoded_image = base64.b64encode(buffer.read()).decode()

      # ---------------- RETURN TO TAMPERMONKEY ----------------
      return jsonify({
          "status": "ok",
          "tags": tags,
          "image": encoded_image,
          "scene": last_scene
      })
    except Exception as e:
      print("ERROR:", e)
      return jsonify({
          "status": "error",
          "message": str(e)
      }), 500
    finally:
      request_active = False
def run_app():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()
public_url = ngrok.connect(5000)
print("COPY THIS LINK:\n", public_url.public_url)


while True:
    time.sleep(60)